# Interval Scheduling Greedy

https://atcoder.jp/contests/abc473/submissions/79043475

**Problem:** given a set of intervals, take as many as possible without overlapping.

---

## The algorithm

> Sort by **right** endpoint. Sweep left to right. Take an interval if it starts at or after where the last taken one ended.

State: one number, `last` = right end of the most recently taken interval.

```
   start ≥ last   →  TAKE, then last = end
   start <  last  →  SKIP
```

---

## `last` is a wall that slides right

```
gaps:     0     1     2     3     4     5     6

                      ║
          ← taken →   ║   ← still free
                      ║
                     last
```

Anything crossing the wall is unavailable. Anything to the right of it is fair game.

---

## The three cases (`last = 2`)

**Starts after the wall → take**

```
gaps:     0     1     2     3     4     5
                      ║
   [3,5)              ║         ●─────●
                      ║         ↑
                     last     start=3 ≥ 2   ✓
```

**Starts exactly at the wall → take**

```
gaps:     0     1     2     3     4     5
                      ║
   [2,4)              ●─────────●
                      ║
                     last     start=2 ≥ 2   ✓
```

Half-open intervals share only a point, no interior:

```
   previous  ├─────────┤
   this      ─────────→├─────────┤
                       ↑
                 shared point only
```

**Starts before the wall → skip**

```
gaps:     0     1     2     3     4     5
                      ║
   [1,4)        ●─────╫─────●
                      ║
                     last     start=1 < 2   ✗
```

**After taking, the wall jumps:**

```
   before:              ║              last = 2
   take [2,4):          ●─────────●
   after:                         ║    last = 4
```

---

## Why sort by RIGHT endpoint

The wall only moves as far right as the interval you take forces it. So always take the one that leaves the wall furthest left — right-endpoint order guarantees that's the first one you can take.

```
   by LEFT end:                          by RIGHT end:

   ●───────────────────────●             ●───────────●
   0           2           4                         ●───────────●
   take → last = 4                       0     2     4
   nothing else fits                     take, take

           ans = 1   ✗                           ans = 2   ✓
```

Same intervals, different order, different answer.

---

## Ties on the right endpoint

Doesn't matter. Two intervals with the same right end always overlap (the shorter sits inside the longer), so you take at most one — and either way `last` lands on the same value.

```
   [1,5)     ●───────────────────●
   [3,5)                 ●───────●     ← nested, mutually exclusive
```

Just `sort by r`, leave the tiebreak unspecified.

---

## If the candidate set is quadratic, prune first

Common trap: the intervals aren't given, they're *generated* by a matching rule — and then there are Θ(N²) of them.

```
gap:      0     1     2     3        4 gaps, all matching
   (0,1)  ●─────●
   (0,2)  ●───────────●
   (0,3)  ●─────────────────●          every PAIR is an interval
   (1,2)        ●─────●                → m(m-1)/2 = 6
   (1,3)        ●───────────●
   (2,3)              ●─────●
```

At `N = 2×10⁵` that's ~2×10¹⁰ intervals. Can't build them.

**Escape:** a long interval is dominated by the short ones inside it.

```
   one long:     ●─────────────────●     scores 1
                 0                 3

   three short:  ●─────●─────●─────●     scores 3
                 0     1     2     3
```

So keep only **each right endpoint back to its nearest matching left endpoint** — one interval per position, O(N) total, same answer.

Look for this whenever the interval set is defined by "all pairs such that…".

---

## Sweep and generate in one pass

Once pruned, you generate intervals in right-endpoint order anyway (you scan left to right), so **no sort is needed** and you never store them:

```cpp
prev[base_value] = 0;                    // base state, present holding 0
for (each position r) {
    v = value_at(r);
    if (prev.contains(v) && last <= prev[v]) {
        ++ans;
        last = r;                        // wall jumps
    }
    prev[v] = r;                         // keep only the nearest match
}
```

---

## Checklist

- Half-open `[l, r)` if touching should be allowed; closed if not.
- Base state must be **in** the table, not implied.
- Candidate set quadratic? Prune by dominance before sorting.
- Sort by right end. Ties don't matter.